# EGC COG Tree

Builds the COG annotation tree and writes one `.cog.json` per sample.

**Run order:** Cell by cell top to bottom. Restart kernel before re-running.

In [1]:
import json
import requests
from pathlib import Path
from collections import defaultdict
from bs4 import BeautifulSoup
import sys
sys.path.append(r"C:\Users\sakoehn\Desktop\JGI\Notebooks")
from All_Coverages import allfold
from weighted_totals import calculate_denominator_egc

## 1. Load sample list

In [2]:
with open(r'C:\Users\sakoehn\Desktop\JGI\sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f"Loaded {len(samples)} samples")

Loaded 128 samples


## 2. Fetch COG metadata from NIH

In [3]:
urls = [
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/J/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/A/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/K/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/L/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/B/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/D/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/Y/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/V/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/T/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/M/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/N/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/Z/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/W/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/U/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/O/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/X/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/C/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/G/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/E/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/F/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/H/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/I/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/P/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/Q/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/R/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/S/",
]

cogs = {}

for url in urls:
    r = requests.get(url)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    cat_letter = url.rstrip("/").split("/")[-1]
    cogs[cat_letter] = {}

    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 5:
            continue
        link_tag = tds[2].find("a")
        if not link_tag:
            continue
        cog_id = link_tag.get_text(strip=True)
        annot  = tds[4].get_text(strip=True)
        cogs[cat_letter][cog_id] = [annot, link_tag.get("href")]

cog_map = {
    cog_id: {"name": name, "category": cat}
    for cat, cog_dict in cogs.items()
    for cog_id, (name, _) in cog_dict.items()
}
print(f"cog_map: {len(cog_map)} entries")

cog_map: 5061 entries


## 3. Read COG annotation files → cog_db

`cog_db[gene_id]` holds parallel lists: one entry per observation of that gene across all samples.

**Fix:** both branches of the if/else write to the same `cog_db` parameter (previously the else branch accidentally wrote to the outer `cog_db` variable, dropping multi-sample entries).

In [4]:
def read_cog(directpath, filename, s, cog_db):
    """
    Parse one sample's .a.cog.txt and append each gene observation
    to cog_db.  The three lists (lookupId, timepoint, scaffolds) are
    parallel: index i of each belongs to the same observation.
    """
    path = directpath / filename
    if not path.exists():
        print(f"  WARNING: {path} not found, skipping")
        return

    with open(path, "r") as f:
        for line in f:
            parts   = line.rstrip("\n").split("\t")
            gene_id = parts[0]
            cog_id  = parts[1]
            scaff   = gene_id.rsplit('_', 2)[0]

            if gene_id not in cog_db:
                cog_db[gene_id] = {
                    'lookupId':  [cog_id],
                    'timepoint': [s],
                    'scaffolds': [scaff],
                }
            else:
                cog_db[gene_id]['lookupId'].append(cog_id)  
                cog_db[gene_id]['timepoint'].append(s)
                cog_db[gene_id]['scaffolds'].append(scaff)


base   = Path(r"C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples")
cog_db = {}

for s in samples:
    read_cog(base / s, f"{s}.a.cog.txt", s, cog_db)

print(f"cog_db: {len(cog_db):,} genes")

cog_db: 50,070,928 genes


## 4. Build COG tree (flat)

COG has no meaningful biological hierarchy — the 25 single-letter functional
categories (C, J, K, …) are labels, not a tree. Each COG entry is registered
once with its category letter stored as metadata.

**Deduplication:** hmmsearch reports one row per domain alignment hit, so a
multi-domain protein can appear multiple times for the same COG in the same
sample. Each gene is counted **once per unique gene_id per (COG, sample)**
using the `genes` set as a gate. This matches JGI's unique-gene counting.

In [5]:
cog_tree = {}  # {cog_id: {name, category, timepoints}}

for gene_id, ginfo in cog_db.items():
    cog_ids    = ginfo['lookupId']
    timepoints = ginfo['timepoint']
    scaffolds  = ginfo['scaffolds']

    for cog_id, sample, scaff in zip(cog_ids, timepoints, scaffolds):
        if cog_id not in cog_map:
            continue

        cat_letter = cog_map[cog_id]['category']
        cog_name   = cog_map[cog_id]['name']

        if cog_id not in cog_tree:
            cog_tree[cog_id] = {
                'name':       cog_name,
                'category':   cat_letter,
                'timepoints': {},
            }
        node = cog_tree[cog_id]
        if sample not in node['timepoints']:
            node['timepoints'][sample] = {'egc': 0, 'norm_egc': 0.0, 'scaffolds': {}, 'genes': set()}
        tp = node['timepoints'][sample]
        if gene_id not in tp['genes']:
            tp['scaffolds'][scaff] = tp['scaffolds'].get(scaff, 0) + 1
        tp['genes'].add(gene_id)

print(f"cog_tree: {len(cog_tree)} COG nodes (flat — no category parent nodes)")

cog_tree: 4612 COG nodes (flat — no category parent nodes)


## 5. Compute EGC and norm_egc

`egc = Σ (gene_count_on_scaffold × scaffold_coverage)`, using only genes annotated with a COG (from this notebook's own `cog_db`).

`norm_egc = egc / weighted_totals[sample]`, where the denominator comes from `Notebooks/weighted_totals.py` (`calculate_denominator_egc`) and is the SAME value used by `EGC_KEGG_TREE.ipynb` / `EGC_ENZYME_TREE.ipynb` / `EGC_TAX_TREE.ipynb` for a given sample -- computed once from every gene with any annotation across all four sources (via the mapping TSVs), not just genes with a COG hit. That shared scope is what makes norm_egc comparable across annotation types on the same plot. norm_egc still stays in (0, 1], since `egc` (a COG-only subset of the shared total) can never exceed it.

In [6]:
# Cross-source-comparable denominator.
#
# calculate_denominator_egc() (Notebooks/weighted_totals.py) computes ONE
# shared total per sample from the mapping TSVs -- every gene annotated by
# ANY of the four sources (taxonomy/KEGG/enzyme/COG), counted once each --
# so this notebook uses the exact same denominator as EGC_KEGG_TREE.ipynb /
# EGC_ENZYME_TREE.ipynb / EGC_TAX_TREE.ipynb for a given sample. That's what
# makes norm_egc comparable across annotation types on the same plot,
# instead of each notebook normalizing against only its own source's
# annotated genes.
weighted_totals = calculate_denominator_egc(samples)

# Sanity check on first sample: compare against the OLD unweighted,
# not-gene-count-weighted sum (allfold[sample]['total']) as a rough
# magnitude check -- not a correctness check on its own.
_s0 = next(iter(weighted_totals))
_new = weighted_totals[_s0]
_old = allfold.get(_s0, {}).get('total', 0)
print(f'Sample {_s0}: old denom={_old:,.0f}  new denom={_new:,.0f}')
print(f'norm_egc values will be ~{_old/_new:.2f}x smaller, now bounded by 1.0')

Sample 3300060358: old denom=8,298,642  new denom=16,137,978
norm_egc values will be ~0.51x smaller, now bounded by 1.0


In [7]:
def avg_gene_map(node_id):
    node = cog_tree[node_id]
    for sample_name, data in node['timepoints'].items():
        scaffold_counts = data.get('scaffolds')
        if not scaffold_counts or sample_name not in allfold:
            continue

        covs = allfold[sample_name]
        sample_total_depth = weighted_totals[sample_name]

        total_egc = 0
        for scaff, cnt in scaffold_counts.items():
            cov = covs.get(scaff, 0)
            total_egc += cnt * cov

        data['egc']      = total_egc
        data['norm_egc'] = (
            total_egc / sample_total_depth
            if sample_total_depth > 0 else 0.0
        )


for node_id in cog_tree:
    avg_gene_map(node_id)
print('EGC computation complete')


EGC computation complete


## 6. Write .cog.json files

In [8]:
base = Path(r"C:\Users\sakoehn\Desktop\JGI\DataStructures\COG")
base.mkdir(parents=True, exist_ok=True)

for s in samples:
    tp_out = {}
    for node_id, node in cog_tree.items():
        if s not in node['timepoints']:
            continue
        tp = node['timepoints'][s]
        tp_out[node_id] = {
            'name':      node['name'],
            'category':  node['category'],
            'egc':       tp.get('egc',       0),
            'norm_egc':  tp.get('norm_egc',  0.0),
            'scaffolds': tp.get('scaffolds', {}),
        }
    with open(base / f"{s}.cog.json", 'w', encoding='utf-8') as f:
        json.dump(tp_out, f, indent=2)

print(f"Wrote {len(samples)} .cog.json files to {base}")

Wrote 128 .cog.json files to C:\Users\sakoehn\Desktop\JGI\DataStructures\COG


In [2]:
import requests

r = requests.get("http://localhost:8000/samples/3300060358/cog", params={"page_size": 10000})
body = r.json()

# Print the raw structure of the first few items
for item in body["data"][:3]:
    print(item)

{'source': 'cog', 'sample': '3300060358', 'timepoint': {'taxon_oid': '3300060358', 'timepoint': '5272121', 'datetime': '5/27/21 21:00'}, 'annotation_id': 'COG0055', 'name': 'COG0055', 'value': {'name': 'COG0055', 'category': 'C', 'egc': 7092.0, 'norm_egc': 0.0008545976558574283, 'scaffolds': {'Ga0590986_0000001': 1, 'Ga0590986_0000003': 1, 'Ga0590986_0000170': 1, 'Ga0590986_0000360': 1, 'Ga0590986_0000370': 1, 'Ga0590986_0000410': 1, 'Ga0590986_0000543': 1, 'Ga0590986_0000579': 1, 'Ga0590986_0000898': 1, 'Ga0590986_0001836': 1, 'Ga0590986_0001956': 1, 'Ga0590986_0002074': 1, 'Ga0590986_0002351': 1, 'Ga0590986_0002596': 1, 'Ga0590986_0004044': 1, 'Ga0590986_0004136': 1, 'Ga0590986_0004549': 1, 'Ga0590986_0005104': 1, 'Ga0590986_0005139': 1, 'Ga0590986_0005182': 1, 'Ga0590986_0005190': 1, 'Ga0590986_0006164': 1, 'Ga0590986_0006965': 1, 'Ga0590986_0007006': 1, 'Ga0590986_0007436': 1, 'Ga0590986_0007625': 1, 'Ga0590986_0007715': 1, 'Ga0590986_0007811': 1, 'Ga0590986_0007834': 1, 'Ga0590986

In [3]:
r2 = requests.get("http://localhost:8000/series/annotations", 
                  params={"source": "cog", "annotation_id": "COG0477"})
for row in r2.json()["data"]:
    if row["sample"] == "3300060358":
        print(row)

{'source': 'cog', 'sample': '3300060358', 'timepoint': {'taxon_oid': '3300060358', 'timepoint': '5272121', 'datetime': '5/27/21 21:00'}, 'annotation_id': 'COG0477', 'name': 'COG0477', 'egc': 74882.0, 'norm_egc': 0.009023404070208114, 'value': {'name': 'COG0477', 'category': 'R', 'egc': 74882.0, 'norm_egc': 0.009023404070208114, 'scaffolds': {'Ga0590986_0000001': 2, 'Ga0590986_0000002': 2, 'Ga0590986_0000005': 2, 'Ga0590986_0000006': 6, 'Ga0590986_0000008': 2, 'Ga0590986_0000010': 2, 'Ga0590986_0000014': 2, 'Ga0590986_0000019': 2, 'Ga0590986_0000029': 2, 'Ga0590986_0000032': 4, 'Ga0590986_0000046': 2, 'Ga0590986_0000054': 2, 'Ga0590986_0000057': 2, 'Ga0590986_0000064': 2, 'Ga0590986_0000066': 4, 'Ga0590986_0000067': 2, 'Ga0590986_0000070': 1, 'Ga0590986_0000071': 2, 'Ga0590986_0000082': 2, 'Ga0590986_0000085': 2, 'Ga0590986_0000089': 1, 'Ga0590986_0000091': 2, 'Ga0590986_0000093': 2, 'Ga0590986_0000094': 2, 'Ga0590986_0000111': 2, 'Ga0590986_0000120': 2, 'Ga0590986_0000127': 2, 'Ga05909

2857
